In [ ]:

dbutils.widgets.text("catalog", "develop")
dbutils.widgets.text("schema", "bronze")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:

from grupo10.utils.logger import get_logger
from grupo10.utils.audit import ensure_audit_table
from grupo10.ingest import ensure_control_table
from databricks.sdk.runtime import spark, dbutils

#Cria as constantes do notebooks
AUDIT_TABLE   = f"{catalog}.{schema}._ingestion_audit"
CONTROL_TABLE = f"{catalog}.{schema}._ingestion_control"
#Instancia o logger e garante existência das tabelas de auditoria e controle
logger = get_logger("sqlserver_ingest_setup")
ensure_audit_table(spark, AUDIT_TABLE)
ensure_control_table(spark, CONTROL_TABLE)
logger.info(f"Audit table ready: {AUDIT_TABLE}")
logger.info(f"Control table ready: {CONTROL_TABLE}")

In [ ]:
import json
import os

#Acessa JSON com as tabelas que devem ser ingeridas e disponibiliza ela para task subsequente
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)

if not notebook_dir.startswith("/Workspace"):
    notebook_dir = "/Workspace" + notebook_dir

tabelas_config_path = os.path.join(notebook_dir, "tables_config.json")

with open(tabelas_config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

target_tables = config["tables"]  # list of {"name": ..., "primary_key": ...}

dbutils.jobs.taskValues.set(key="tables", value=target_tables)
